In [ ]:
# Reddit Data Workflow
### Woojin Park, Korea University
### Copyright 2026, Korea University

# 00. Reddit Subreddit Data Preparation

This notebook is the first step of the project pipeline. It mirrors the original subreddit-specific notebooks such as `reddit_data_preprocessing_depression.ipynb`, `reddit_data_preprocessing_AnxietyDepression.ipynb`, and related files, but consolidates the repeated logic into one notebook.

The goal of this step is to create subreddit-level preprocessed parquet files used by `01_reddit_data_preprocessing.ipynb`.

## Methodology 1. Reddit PRAW API

The original notebooks kept this section as a reference for small API-based collection. The main dataset used in this project is based on Reddit archive dumps, so the PRAW block remains commented out.

In [ ]:
# ### put your personal info
# import praw
# reddit = praw.Reddit(client_id='',
#                      client_secret='',
#                      user_agent='')
#
# posts = []
# ml_subreddit = reddit.subreddit('depression')
# for post in ml_subreddit.new(limit=1000):
#     posts.append([post.title, post.score, post.id, post.subreddit, post.url,
#                   post.num_comments, post.selftext, post.created])

## Methodology 2. Reddit Archive Data Dump

The original notebooks loaded subreddit archive dumps from local `.zst` files when available, converted `created_utc`, saved raw parquet files, selected common columns, and saved subreddit-level preprocessed parquet files.

This integrated notebook keeps that same shape, but uses project-relative paths.

In [ ]:
# Run once if dependencies are missing.
# %pip install -r ../requirements.txt

In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import zstandard as zstd

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'reddit_archives'
RAW_PARQUET_DIR = PROJECT_ROOT / 'data' / 'interim' / 'subreddit_raw'
PREPROCESSED_DIR = PROJECT_ROOT / 'data' / 'interim' / 'subreddit_preprocessed'

RAW_PARQUET_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('RAW_DIR:', RAW_DIR)
print('RAW_PARQUET_DIR:', RAW_PARQUET_DIR)
print('PREPROCESSED_DIR:', PREPROCESSED_DIR)

In [ ]:
# Source configuration based on the original subreddit-specific notebooks.
# If a .zst source is missing but a raw parquet or preprocessed parquet already exists,
# the notebook can still continue from the available local file.
SOURCES = [
    dict(name='AnxietyDepression', variable='anxiety_depression_submissions', zst='AnxietyDepression_submissions.zst', raw_parquet='anxiety_depression_submissions.parquet', preprocessed='anxiety_depression_submissions_preprocessed.parquet', nrows=2_000_000),
    dict(name='depression', variable='depression_submissions', zst='depression_submissions.zst', raw_parquet='depression_submissions.parquet', preprocessed='depression_submissions_preprocessed.parquet', nrows=300_000),
    dict(name='technology', variable='technology_submissions', zst='technology_submissions.zst', raw_parquet='technology_submissions.parquet', preprocessed='technology_submissions_preprocessed.parquet', nrows=600_000),
    dict(name='AskScienceDiscussion', variable='askscience_discussion_submissions', zst='askscience_discussion_submissions.zst', raw_parquet='askscience_discussion_submissions.parquet', preprocessed='askscience_discussion_submissions_preprocessed.parquet', nrows=1_000_000),
    dict(name='webdev', variable='webdev_discussion_submissions', zst='webdev_submissions.zst', raw_parquet='webdev_discussion_submissions.parquet', preprocessed='webdev_discussion_submissions_preprocessed.parquet', nrows=1_000_000),
    dict(name='datascience', variable='datascience_submissions', zst='datascience_submissions.zst', raw_parquet='datascience_submissions.parquet', preprocessed='datascience_submissions_preprocessed.parquet', nrows=4_000_000),
    dict(name='Positivity', variable='positivity_submissions', zst='Positivity_submissions.zst', raw_parquet='positivity_submissions.parquet', preprocessed='positivity_submissions_preprocessed.parquet', nrows=4_000_000),
    dict(name='MadeMeSmile', variable='mademesmile_submissions', zst='MadeMeSmile_submissions.zst', raw_parquet='mademesmile_submissions.parquet', preprocessed='mademesmile_submissions_preprocessed.parquet', nrows=100_000),
    dict(name='UnexpectedlyWholesome', variable='unexpectedlyWholesome_submissions', zst='UnexpectedlyWholesome_submissions.zst', raw_parquet='unexpectedlyWholesome_submissions.parquet', preprocessed='unexpectedlyWholesome_submissions_preprocessed.parquet', nrows=100_000),
    dict(name='CongratsLikeImFive', variable='congrats_submissions', zst='CongratsLikeImFive_submissions.zst', raw_parquet='congrats_submissions.parquet', preprocessed='congrats_submissions_preprocessed.parquet', nrows=30_000),
    dict(name='happy', variable='happy_submissions', zst='happy_submissions.zst', raw_parquet='happy_submissions.parquet', preprocessed='happy_submissions_preprocessed.parquet', nrows=2_000_000),
]

KEEP_COLUMNS = ['subreddit', 'author', 'over_18', 'link_flair_text', 'title', 'selftext', 'url', 'created_utc']

pd.DataFrame(SOURCES)

In [ ]:
sorted([p.name for p in RAW_DIR.glob('*')])

## Helper Functions

In [ ]:
def convert_created_utc(df):
    df = df.copy()
    if 'created_utc' in df.columns:
        df['created_utc'] = df['created_utc'].apply(
            lambda x: datetime.utcfromtimestamp(x) if isinstance(x, (int, float)) else x
        )
    return df


def load_raw_source(source):
    zst_path = RAW_DIR / source['zst'] if source.get('zst') else None
    raw_parquet_path = RAW_DIR / source['raw_parquet']
    existing_raw_parquet_path = RAW_PARQUET_DIR / source['raw_parquet']
    existing_preprocessed_path = PREPROCESSED_DIR / source['preprocessed']

    if zst_path and zst_path.exists():
        df = pd.read_json(
            zst_path,
            compression=dict(method='zstd', max_window_size=2147483648),
            lines=True,
            nrows=source['nrows'],
        )
        return convert_created_utc(df), 'zst'

    if raw_parquet_path.exists():
        return pd.read_parquet(raw_parquet_path, engine='pyarrow'), 'raw_parquet'

    if existing_raw_parquet_path.exists():
        return pd.read_parquet(existing_raw_parquet_path, engine='pyarrow'), 'existing_raw_parquet'

    if existing_preprocessed_path.exists():
        return pd.read_parquet(existing_preprocessed_path, engine='pyarrow'), 'existing_preprocessed_only'

    raise FileNotFoundError(f"No source file found for {source['name']}")


def make_preprocessed(df):
    missing = [column for column in KEEP_COLUMNS if column not in df.columns]
    if missing:
        raise KeyError(f'Missing expected columns: {missing}')
    return df[KEEP_COLUMNS].copy()

## Step 1. Load Archive/Raw Data and Save Subreddit-Level Preprocessed Parquet

This replaces the repeated cells in the original subreddit-specific notebooks:

1. Load downloaded submission dump.
2. Convert timestamp to UTC.
3. Save raw parquet for faster loading.
4. Select common modeling columns.
5. Save final subreddit-level preprocessed parquet.

In [ ]:
summary = []

for source in SOURCES:
    print(f"
=== {source['name']} ===")
    df, load_status = load_raw_source(source)
    print('Load status:', load_status)
    print('Raw shape:', df.shape)

    # step2. download/save the raw parquet file to speed up later-stage data loading
    raw_out = RAW_PARQUET_DIR / source['raw_parquet']
    if load_status != 'existing_preprocessed_only':
        df.to_parquet(raw_out, engine='pyarrow')

    # step3. select common columns and save final preprocessed data into parquet
    preprocessed_df = make_preprocessed(df)
    preprocessed_out = PREPROCESSED_DIR / source['preprocessed']
    preprocessed_df.to_parquet(preprocessed_out, engine='pyarrow')

    print('Preprocessed shape:', preprocessed_df.shape)
    display(preprocessed_df.head(2))

    summary.append({
        'subreddit': source['name'],
        'load_status': load_status,
        'raw_records': len(df),
        'preprocessed_records': len(preprocessed_df),
        'preprocessed_file': source['preprocessed'],
    })

## Step 2. Preparation Summary

In [ ]:
preparation_summary = pd.DataFrame(summary)
preparation_summary

In [ ]:
summary_path = PREPROCESSED_DIR / 'subreddit_preparation_summary.csv'
preparation_summary.to_csv(summary_path, index=False)
summary_path

## Step 3. Quality Check

The next notebook, `01_reddit_data_preprocessing.ipynb`, expects these subreddit-level preprocessed parquet files under `data/interim/subreddit_preprocessed/`.

In [ ]:
sorted([p.name for p in PREPROCESSED_DIR.glob('*_preprocessed.parquet')])